# Dual-Mode Orchestration

The same agent can have two "brains": `on_agent` lets the LLM freely explore and decide, while `on_workflow` gives you precise, deterministic control over every step. Understanding how to write both modes is the prerequisite for mastering the amphibious design.

In this tutorial, we'll build an **e-commerce price monitor** — using Agent mode to let the LLM discover optimal comparison strategies, and Workflow mode to run a fixed price-checking pipeline.

## Initialize

First, let's set up the LLM and define the tools our price monitor will use.

In [ ]:
import os
from bridgic.model import OpenAILlm

api_key = os.environ.get("OPENAI_API_KEY", "your-api-key")
base_url = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")

llm = OpenAILlm(model="gpt-4o-mini", api_key=api_key, base_url=base_url)

In [ ]:
from bridgic.core import tool


@tool(description="Search for product prices on a specific platform")
async def search_price(platform: str, product: str) -> str:
    prices = {
        ("amazon", "laptop"): "$999",
        ("ebay", "laptop"): "$879",
        ("walmart", "laptop"): "$949",
        ("amazon", "headphones"): "$199",
        ("ebay", "headphones"): "$159",
        ("walmart", "headphones"): "$179",
    }
    price = prices.get((platform.lower(), product.lower()), "Price not found")
    return f"{product} on {platform}: {price}"


@tool(description="Compare prices across multiple results and find the best deal")
async def compare_prices(price_list: str) -> str:
    return f"Best deal analysis based on: {price_list}"


@tool(description="Generate a price comparison report")
async def generate_report(product: str, findings: str) -> str:
    return f"=== Price Report for {product} ===\n{findings}\nReport generated successfully."

## Part 1: Agent Mode — LLM-Driven Orchestration

In `on_agent()`, the LLM has full autonomy. You define *what to think about* using think units, and the LLM decides which tools to call, in what order, and how to combine results.

A `think_unit` wraps a `CognitiveWorker` — an instruction that tells the LLM what goal to pursue. The LLM then reasons about which tools to invoke and how to chain their outputs together.

In [ ]:
from bridgic.amphibious import AmphibiousAutoma, CognitiveContext, CognitiveWorker, think_unit


class PriceMonitorAgent(AmphibiousAutoma[CognitiveContext]):
    researcher = think_unit(
        CognitiveWorker.inline(
            "Search for product prices across different platforms. "
            "Try to cover as many platforms as possible for a comprehensive comparison."
        ),
        max_attempts=8,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.researcher

In [ ]:
agent = PriceMonitorAgent(llm=llm)
result = await agent.arun(
    goal="Find the best deal for a laptop across Amazon, eBay, and Walmart. Compare prices and generate a report.",
    tools=[search_price, compare_prices, generate_report],
)
print(result)

Notice how the LLM autonomously chose which platforms to search, decided when enough data was collected, and generated the report — all without any hardcoded flow.

### Combining Multiple Think Units

You can define multiple think units to break the agent's reasoning into distinct phases. Each phase focuses on a specific sub-goal, and `self.sequential()` scopes the context so that one phase's results feed into the next.

In [ ]:
class MultiPhaseMonitor(AmphibiousAutoma[CognitiveContext]):
    scanner = think_unit(
        CognitiveWorker.inline("Search for the product price on each available platform."),
        max_attempts=5,
    )
    analyst = think_unit(
        CognitiveWorker.inline("Compare all collected prices and generate a final report."),
        max_attempts=3,
    )

    async def on_agent(self, ctx: CognitiveContext):
        async with self.sequential(goal="Scan all platforms for prices"):
            await self.scanner
        async with self.sequential(goal="Analyze results and generate report"):
            await self.analyst

> `self.sequential()` is a **Phase Annotation** that scopes context — we'll cover this in detail in the Phase Annotation tutorial.

## Part 2: Workflow Mode — Deterministic Step-by-Step Execution

In `on_workflow()`, **you** define every step. The method is an async generator — each `yield step(...)` pauses execution, runs the specified tool, and returns the result. There is no LLM reasoning involved in deciding what to do next; the flow is entirely determined by your code.

In [ ]:
from bridgic.amphibious import step


class PriceMonitorWorkflow(AmphibiousAutoma[CognitiveContext]):
    async def on_agent(self, ctx: CognitiveContext):
        pass  # Required but not used in pure workflow mode

    async def on_workflow(self, ctx: CognitiveContext):
        # Step 1: Search each platform
        amazon_price = yield step("search_price", platform="Amazon", product="laptop")
        ebay_price = yield step("search_price", platform="eBay", product="laptop")
        walmart_price = yield step("search_price", platform="Walmart", product="laptop")

        # Step 2: Compare results
        all_prices = f"{amazon_price}, {ebay_price}, {walmart_price}"
        comparison = yield step("compare_prices", price_list=all_prices)

        # Step 3: Generate report
        yield step("generate_report", product="laptop", findings=str(comparison))

In [ ]:
workflow = PriceMonitorWorkflow(llm=llm)
result = await workflow.arun(
    goal="Compare laptop prices across platforms",
    tools=[search_price, compare_prices, generate_report],
)
print(result)

Every step executes in exactly the order you defined. The result of each `yield step()` is available for subsequent steps. No LLM reasoning overhead — pure deterministic execution.

### AgentFallback — Delegating to the LLM Mid-Workflow

Sometimes a workflow encounters a situation too complex for a fixed step. `AgentFallback` lets you hand control to the agent mode temporarily — the LLM takes over, reasons about the current state, and acts accordingly. Once it finishes, control returns to the workflow.

In [ ]:
from bridgic.amphibious import AgentFallback


class HybridMonitor(AmphibiousAutoma[CognitiveContext]):
    helper = think_unit(
        CognitiveWorker.inline("Analyze the situation and decide the best course of action."),
        max_attempts=5,
    )

    async def on_agent(self, ctx: CognitiveContext):
        await self.helper

    async def on_workflow(self, ctx: CognitiveContext):
        # Fixed steps for known platforms
        amazon_price = yield step("search_price", platform="Amazon", product="headphones")
        ebay_price = yield step("search_price", platform="eBay", product="headphones")

        # Delegate to agent for complex analysis
        yield AgentFallback(
            goal="Analyze the collected prices and determine if we should check more platforms or generate the report now.",
            max_attempts=3,
        )

## Which Mode Should You Choose?

| Aspect | Agent Mode (`on_agent`) | Workflow Mode (`on_workflow`) |
|--------|------------------------|------------------------------|
| **Decision Maker** | LLM | Developer's code |
| **How to Define** | `await self.think_unit` | `yield step(...)` |
| **Best For** | Open-ended, exploratory tasks | Known, repeatable procedures |
| **Predictability** | Lower (LLM may vary each run) | High (steps are fixed) |
| **Flexibility** | High (adapts to unforeseen situations) | Lower (follows predefined path) |
| **LLM Cost** | Higher (reasoning at each step) | Lower (no reasoning for flow control) |

In practice, you rarely need to choose just one — the amphibious design lets you combine both in a single agent, which we'll explore in the next tutorial.

<div style="text-align: center; margin: 2rem 0;">
<hr style="border: none; border-top: 2px solid #e2e8f0;">
</div>

## What have we learnt?

In this tutorial, we built a price monitoring system using both orchestration modes:

- **`on_agent(ctx)`** is the LLM-driven mode. Inside it, you `await` think units. The LLM autonomously decides which tools to call and in what order. Use `self.sequential()` or `self.loop()` to organize execution into phases.
- **`on_workflow(ctx)`** is the deterministic mode. It's an async generator where each `yield step("tool_name", **args)` executes a specific tool and returns the result. Steps run in exactly the order you define.
- **`AgentFallback`** can be yielded inside `on_workflow` to temporarily hand control to the agent mode for complex sub-tasks.
- **Both modes coexist** in the same `AmphibiousAutoma` class. This duality is what makes the framework "amphibious."

Next, we'll see how these two modes work together at runtime — with automatic switching and graceful degradation.